# Screening abnormal price-volume emiten IDX

Pipeline riset yang *reproducible*: ±960 emiten, 5 tahun data harian dari `yfinance`, target *future 20-trading-day abnormal price-volume event*, nonnegative weighted score (jumlah bobot = 1), Optuna dengan nested walk-forward validation, serta holdout 2026 yang tidak disentuh optimasi.

> **Anti-leakage:** seluruh fitur memakai informasi sampai tanggal scoring; label melihat 20 hari berikutnya; split diberi embargo 20 hari; 2026 tidak boleh dipakai memilih bobot atau threshold.

In [1]:
# Jalankan sekali bila environment belum siap:
# %pip install -q -r requirements.txt

from pathlib import Path
import json, math, warnings
from datetime import date

import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm
import optuna
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
SEED = 42
rng = np.random.default_rng(SEED)

## 1. Konfigurasi

`NOTEBOOK_DIR` dibuat tahan terhadap notebook yang dibuka dari root proyek maupun dari folder ini. Batas target default adalah hipotesis riset, bukan fakta universal; lakukan sensitivity analysis tanpa menyentuh holdout 2026.

In [2]:
cwd = Path.cwd().resolve()
NOTEBOOK_DIR = cwd if (cwd / 'config').exists() else cwd / 'Model Screening UMA'
ROOT = NOTEBOOK_DIR.parent
UNIVERSE_XLSX = ROOT / 'Analisis Pendahuluan' / 'KODE.xlsx'
CONTROL_CSV = NOTEBOOK_DIR / 'config' / 'control_universe.csv'
UMA_CSV = NOTEBOOK_DIR / 'config' / 'official_uma.csv'
RAW_DIR = NOTEBOOK_DIR / 'data' / 'raw'
PROCESSED_DIR = NOTEBOOK_DIR / 'data' / 'processed'
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs'
for p in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]: p.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp.today().normalize()
FINAL_TEST_START = pd.Timestamp('2026-01-01')
END_DATE = TODAY + pd.Timedelta(days=1)  # yfinance end bersifat exclusive
START_DATE = END_DATE - pd.DateOffset(years=5) - pd.Timedelta(days=45)
BENCHMARK = '^JKSE'
HORIZON = 20
TOP_K = 50
EMBARGO_DAYS = HORIZON
BATCH_SIZE = 80
FORCE_DOWNLOAD = False
N_TRIALS = 150

# Definisi event: dalam 20 hari ke depan, keduanya terjadi.
TARGET_ABNORMAL_RETURN = 0.10       # max cumulative stock return minus IHSG >= 10%
TARGET_MAX_VOLUME_RATIO = 2.50      # max volume / trailing median volume >= 2.5x
MIN_HISTORY = 120

FEATURES = [
    'mom_5', 'mom_20', 'mom_60', 'abret_5', 'abret_20',
    'volatility_20', 'volume_ratio_5_60', 'volume_z_20',
    'turnover_accel', 'range_20', 'drawdown_60'
]
print({'universe': str(UNIVERSE_XLSX), 'start': str(START_DATE.date()),
       'end': str(END_DATE.date()), 'final_test_start': str(FINAL_TEST_START.date())})

{'universe': '/Users/mraffyzeidan/Downloads/ISTL/Analisis Pendahuluan/KODE.xlsx', 'start': '2021-06-16', 'end': '2026-07-31', 'final_test_start': '2026-01-01'}


## 2. Universe dan data kontrol

IDX80/LQ45 tetap ada dalam training dan scoring. Flag kontrol hanya dipakai untuk mengeluarkan saham tersebut dari tabel kandidat final. File kontrol kosong akan memicu peringatan keras.

In [3]:
universe = pd.read_excel(UNIVERSE_XLSX, sheet_name=0)
universe.columns = universe.columns.str.strip()
universe = universe.rename(columns={'KODE': 'ticker', 'Sektor': 'sector', 'SubSektor': 'subsector'})
universe['ticker'] = universe['ticker'].astype(str).str.strip().str.upper().str.replace('.JK', '', regex=False)
universe = universe[universe['ticker'].str.match(r'^[A-Z0-9]+$')].drop_duplicates('ticker').reset_index(drop=True)
universe['yf_ticker'] = universe['ticker'] + '.JK'

controls = pd.read_csv(CONTROL_CSV)
controls['ticker'] = controls['ticker'].astype(str).str.strip().str.upper().str.replace('.JK', '', regex=False)
controls = controls[controls['ticker'].ne('') & controls['ticker'].ne('NAN')].copy()
for c in ['effective_from', 'effective_to']:
    controls[c] = pd.to_datetime(controls[c], errors='coerce')
control_tickers = set(controls['ticker'])
if not control_tickers:
    warnings.warn('control_universe.csv masih kosong. Isi riwayat IDX80/LQ45 sebelum hasil dipakai secara operasional.')

print(f'{len(universe):,} emiten | {len(control_tickers):,} ticker kontrol')
display(universe.head(), controls.head())

959 emiten | 0 ticker kontrol


/var/folders/m3/3gmfjhln13qbt6p_47473gs40000gn/T/ipykernel_19355/3529722015.py:15: UserWarning: control_universe.csv masih kosong. Isi riwayat IDX80/LQ45 sebelum hasil dipakai secara operasional.
  warnings.warn('control_universe.csv masih kosong. Isi riwayat IDX80/LQ45 sebelum hasil dipakai secara operasional.')


,ticker,sector,subsector,Error,yf_ticker
0,AADI,Energi,"Minyak, Gas & Batu Bara",NaN,AADI.JK
1,AALI,Barang Konsumen Primer,Makanan & Minuman,NaN,AALI.JK
2,ABBA,Barang Konsumen Non-Primer,Media & Hiburan,NaN,ABBA.JK
3,ABDA,Keuangan,Asuransi,NaN,ABDA.JK
4,ABMM,Energi,"Minyak, Gas & Batu Bara",NaN,ABMM.JK


,ticker,index_name,effective_from,effective_to


## 3. Unduh 5 tahun data dengan `yfinance` + `tqdm`

Pengunduhan dilakukan per batch agar cepat dan tetap terlihat progresnya. Hasil dicache; ticker gagal dicatat dan dapat diulang dengan `FORCE_DOWNLOAD=True`.

In [4]:
def download_batch(symbols, start, end):
    raw = yf.download(
        tickers=symbols, start=start, end=end, interval='1d', auto_adjust=False,
        actions=False, group_by='ticker', threads=True, progress=False, timeout=30
    )
    frames = []
    if raw.empty:
        return frames
    if not isinstance(raw.columns, pd.MultiIndex):
        raw.columns = pd.MultiIndex.from_product([[symbols[0]], raw.columns])
    level0 = set(raw.columns.get_level_values(0))
    for symbol in symbols:
        if symbol not in level0:
            continue
        x = raw[symbol].copy().dropna(how='all')
        if x.empty:
            continue
        x.columns = x.columns.str.lower().str.replace(' ', '_')
        x = x.reset_index().rename(columns={'Date': 'date', 'index': 'date'})
        x['ticker'] = symbol.removesuffix('.JK')
        frames.append(x)
    return frames

cache_path = RAW_DIR / f'prices_{START_DATE:%Y%m%d}_{END_DATE:%Y%m%d}.parquet'
if cache_path.exists() and not FORCE_DOWNLOAD:
    prices = pd.read_parquet(cache_path)
else:
    symbols = universe['yf_ticker'].tolist() + [BENCHMARK]
    batches = [symbols[i:i+BATCH_SIZE] for i in range(0, len(symbols), BATCH_SIZE)]
    frames = []
    for batch in tqdm(batches, desc='Download batch', unit='batch'):
        frames.extend(download_batch(batch, START_DATE, END_DATE))
    prices = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if prices.empty:
        raise RuntimeError('Tidak ada data yang berhasil diunduh.')
    prices['date'] = pd.to_datetime(prices['date']).dt.tz_localize(None)
    prices.to_parquet(cache_path, index=False)

prices['date'] = pd.to_datetime(prices['date']).dt.tz_localize(None)
prices = prices.sort_values(['ticker', 'date']).drop_duplicates(['ticker', 'date'])
downloaded = set(prices['ticker'])
missing = sorted(set(universe['ticker']) - downloaded)
pd.DataFrame({'ticker': missing}).to_csv(OUTPUT_DIR / 'missing_yfinance.csv', index=False)
print(f'Rows: {len(prices):,} | ticker berhasil: {len(downloaded):,} | gagal: {len(missing):,}')
display(pd.DataFrame({'ticker_gagal': missing}).head(20))

Download batch:   0%|          | 0/12 [00:00<?, ?batch/s]

$GOTOM.JK: possibly delisted; no timezone found

1 Failed download:
['GOTOM.JK']: possibly delisted; no timezone found


Rows: 1,022,421 | ticker berhasil: 959 | gagal: 1


,ticker_gagal
0,GOTOM


## 4. Feature engineering dan target

Semua rolling statistic digeser secara alami karena baris tanggal *t* hanya memakai data sampai penutupan *t*. Target memakai maksimum abnormal return dan volume ratio pada hari bursa `t+1 ... t+20`. Baris yang belum memiliki horizon penuh diberi label `NaN`, bukan nol.

In [5]:
benchmark_symbol = BENCHMARK.removesuffix('.JK')
# yfinance mempertahankan ^JKSE sebagai ticker; fallback aman untuk batch satu simbol.
bench_rows = prices[prices['ticker'].isin([BENCHMARK, benchmark_symbol, '^JKSE'])].copy()
if bench_rows.empty:
    b = yf.download(BENCHMARK, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
    b = b.reset_index()
    if isinstance(b.columns, pd.MultiIndex): b.columns = b.columns.get_level_values(0)
    bench = b[['Date', 'Close']].rename(columns={'Date': 'date', 'Close': 'bench_close'})
else:
    bench = bench_rows[['date', 'close']].rename(columns={'close': 'bench_close'}).drop_duplicates('date')
bench['date'] = pd.to_datetime(bench['date']).dt.tz_localize(None)

stocks = prices[prices['ticker'].isin(set(universe['ticker']))].copy()
stocks = stocks.merge(bench, on='date', how='left').merge(
    universe[['ticker', 'sector', 'subsector']], on='ticker', how='left'
)

def future_rolling_max(s, horizon):
    # shift(-horizon) + rolling memberi jendela t+1..t+horizon tanpa loop Python.
    return s.iloc[::-1].rolling(horizon, min_periods=horizon).max().iloc[::-1].shift(-1)

def engineer_one(g):
    g = g.sort_values('date').copy()
    px = g['adj_close'].fillna(g['close']) if 'adj_close' in g else g['close']
    ret = px.pct_change()
    bret = g['bench_close'].pct_change()
    g['mom_5'] = px.pct_change(5)
    g['mom_20'] = px.pct_change(20)
    g['mom_60'] = px.pct_change(60)
    g['abret_5'] = g['mom_5'] - g['bench_close'].pct_change(5)
    g['abret_20'] = g['mom_20'] - g['bench_close'].pct_change(20)
    g['volatility_20'] = ret.rolling(20).std() * np.sqrt(252)
    vol_med_20 = g['volume'].rolling(20).median().replace(0, np.nan)
    vol_med_60 = g['volume'].rolling(60).median().replace(0, np.nan)
    g['volume_ratio_5_60'] = g['volume'].rolling(5).mean() / vol_med_60
    g['volume_z_20'] = (g['volume'] - g['volume'].rolling(20).mean()) / g['volume'].rolling(20).std()
    turnover = px * g['volume']
    g['turnover_accel'] = turnover.rolling(5).mean() / turnover.rolling(60).median()
    g['range_20'] = g['high'].rolling(20).max() / g['low'].rolling(20).min() - 1
    g['drawdown_60'] = px / px.rolling(60).max() - 1

    future_px_max = future_rolling_max(px, HORIZON)
    future_bench_max = future_rolling_max(g['bench_close'], HORIZON)
    future_abret = (future_px_max / px - 1) - (future_bench_max / g['bench_close'] - 1)
    future_vol_ratio = future_rolling_max(g['volume'], HORIZON) / vol_med_60
    has_full_horizon = g['date'].shift(-HORIZON).notna()
    g['future_max_abret_20'] = future_abret
    g['future_max_volume_ratio_20'] = future_vol_ratio
    g['target'] = np.where(
        has_full_horizon,
        ((future_abret >= TARGET_ABNORMAL_RETURN) &
         (future_vol_ratio >= TARGET_MAX_VOLUME_RATIO)).astype(float),
        np.nan
    )
    return g

parts = []
for _, g in tqdm(stocks.groupby('ticker', sort=False), total=stocks['ticker'].nunique(), desc='Features', unit='ticker'):
    parts.append(engineer_one(g))
panel = pd.concat(parts, ignore_index=True)
panel = panel.replace([np.inf, -np.inf], np.nan)
panel.to_parquet(PROCESSED_DIR / 'feature_panel.parquet', index=False)
print(panel[['date','ticker','target']].dropna().shape, 'positive rate =', panel['target'].mean())

Features:   0%|          | 0/958 [00:00<?, ?ticker/s]

(1002810, 3) positive rate = 0.18433102980624444


## 5. Cross-sectional transform dan metrik

Model utama sengaja transparan: fitur diubah menjadi percentile rank `[0,1]` per tanggal, arah fitur dipelajari sebagai tanda tetap dari training fold, lalu skor adalah kombinasi nonnegative dengan bobot berjumlah satu. Tidak ada fitting model nonlinear.

In [6]:
def add_cross_sectional_ranks(df, features):
    out = df.copy()
    for f in tqdm(features, desc='Cross-sectional ranks', unit='fitur'):
        out[f'r_{f}'] = out.groupby('date')[f].rank(pct=True, method='average')
    return out

panel = add_cross_sectional_ranks(panel, FEATURES)
RANK_FEATURES = [f'r_{f}' for f in FEATURES]

def precision_at_k(y, score, k=50):
    x = pd.DataFrame({'y': y, 's': score}).dropna().nlargest(k, 's')
    return float(x['y'].mean()) if len(x) else np.nan

def ndcg_at_k(y, score, k=50):
    x = pd.DataFrame({'y': y, 's': score}).dropna().sort_values('s', ascending=False).head(k)
    if x.empty: return np.nan
    gains = x['y'].to_numpy()
    discounts = 1 / np.log2(np.arange(2, len(x) + 2))
    dcg = np.sum(gains * discounts)
    ideal = np.sort(gains)[::-1]
    idcg = np.sum(ideal * discounts)
    return float(dcg / idcg) if idcg > 0 else 0.0

def datewise_topk_metrics(df, k=50):
    rows = []
    for d, x in df.groupby('date'):
        if x['target'].notna().sum() == 0: continue
        p = precision_at_k(x['target'], x['score'], k)
        n = ndcg_at_k(x['target'], x['score'], k)
        base = x['target'].mean()
        rows.append((d, p, n, p / base if base > 0 else np.nan))
    return pd.DataFrame(rows, columns=['date','precision_at_50','ndcg_at_50','lift_at_50'])

def score_frame(df, weights, directions):
    x = df.copy()
    z = x[RANK_FEATURES].fillna(0.5).to_numpy()
    z = np.where(np.asarray(directions)[None, :] > 0, z, 1-z)
    x['score'] = z @ np.asarray(weights)
    return x

def learn_directions(train):
    # Arah hanya dari training window; korelasi nol diarahkan positif.
    corr = train[RANK_FEATURES].corrwith(train['target'], method='spearman').fillna(0)
    return np.where(corr.to_numpy() >= 0, 1, -1)

def evaluate(df):
    x = df.dropna(subset=['target','score'])
    pr = average_precision_score(x['target'], x['score']) if x['target'].nunique() > 1 else np.nan
    daily = datewise_topk_metrics(x, TOP_K)
    return {'pr_auc': pr, 'ndcg_at_50': daily['ndcg_at_50'].mean(),
            'precision_at_50': daily['precision_at_50'].mean(),
            'lift_at_50': daily['lift_at_50'].mean()}

model_data = panel.dropna(subset=['target']).copy()
print(model_data['date'].min(), model_data['date'].max(), len(model_data))

Cross-sectional ranks:   0%|          | 0/11 [00:00<?, ?fitur/s]

2021-06-16 00:00:00 2026-07-02 00:00:00 1002810


## 6. Nested walk-forward + Optuna

Outer folds memberi estimasi performa yang tidak bias. Di setiap outer fold, Optuna hanya melihat inner folds yang seluruhnya berada sebelum outer validation. Objective persis:

`0.50*mean_pr_auc + 0.30*mean_ndcg@50 + 0.20*mean_precision@50 - 0.10*std(fold_pr_auc)`

In [7]:
def make_walk_forward_folds(dates, n_folds=4, min_train_months=18, valid_months=4, embargo_days=20):
    dates = pd.DatetimeIndex(sorted(pd.Series(dates).dropna().unique()))
    first = dates.min() + pd.DateOffset(months=min_train_months)
    last_start = dates.max() - pd.DateOffset(months=valid_months)
    starts = pd.date_range(first, last_start, periods=n_folds)
    folds = []
    for vs in starts:
        vs = dates[dates.searchsorted(vs)] if dates.searchsorted(vs) < len(dates) else None
        if vs is None: continue
        ve_target = vs + pd.DateOffset(months=valid_months)
        ve_idx = min(dates.searchsorted(ve_target), len(dates)-1)
        ve = dates[ve_idx]
        train_end = vs - pd.offsets.BDay(embargo_days)
        folds.append((dates.min(), train_end, vs, ve))
    return folds

def sample_simplex(trial, n):
    # Softmax parameterization menjamin nonnegative dan sum(weights)=1.
    logits = np.array([trial.suggest_float(f'logit_{i}', -4, 4) for i in range(n)])
    logits -= logits.max()
    w = np.exp(logits)
    return w / w.sum()

def optimize_weights(train_pool, inner_folds, n_trials=N_TRIALS):
    def objective(trial):
        weights = sample_simplex(trial, len(RANK_FEATURES))
        metrics = []
        for tr_start, tr_end, va_start, va_end in inner_folds:
            tr = train_pool[(train_pool.date >= tr_start) & (train_pool.date <= tr_end)]
            va = train_pool[(train_pool.date >= va_start) & (train_pool.date <= va_end)]
            if tr.empty or va.empty: continue
            directions = learn_directions(tr)
            metrics.append(evaluate(score_frame(va, weights, directions)))
        if len(metrics) < 2: return -1.0
        m = pd.DataFrame(metrics)
        return (0.50*m.pr_auc.mean() + 0.30*m.ndcg_at_50.mean()
                + 0.20*m.precision_at_50.mean() - 0.10*m.pr_auc.std(ddof=0))

    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study, sample_simplex(study.best_trial, len(RANK_FEATURES))

development = model_data[model_data['date'] < FINAL_TEST_START].copy()
outer_folds = make_walk_forward_folds(development['date'], n_folds=4, min_train_months=24, valid_months=4)
outer_results, outer_weights = [], []

for fold_id, (tr_start, tr_end, va_start, va_end) in enumerate(tqdm(outer_folds, desc='Outer folds'), 1):
    outer_train = development[(development.date >= tr_start) & (development.date <= tr_end)]
    outer_valid = development[(development.date >= va_start) & (development.date <= va_end)]
    inner = make_walk_forward_folds(outer_train['date'], n_folds=3, min_train_months=12, valid_months=3)
    study, weights = optimize_weights(outer_train, inner)
    directions = learn_directions(outer_train)
    met = evaluate(score_frame(outer_valid, weights, directions))
    met.update({'fold': fold_id, 'train_end': tr_end, 'valid_start': va_start,
                'valid_end': va_end, 'inner_best_objective': study.best_value})
    outer_results.append(met)
    outer_weights.append(weights)

outer_results = pd.DataFrame(outer_results)
display(outer_results)
print('Outer PR-AUC instability:', outer_results['pr_auc'].std(ddof=0))

Outer folds:   0%|          | 0/4 [00:00<?, ?it/s]

[I 2026-07-30 09:27:00,466] A new study created in memory with name: no-name-18f094e4-7326-4fa9-a2e1-3658a14c10cb


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-07-30 09:27:01,930] Trial 0 finished with value: 0.3773051008529895 and parameters: {'logit_0': -1.0036790492211, 'logit_1': 3.6057144512793293, 'logit_2': 1.8559515344912407, 'logit_3': 0.7892678735762928, 'logit_4': -2.751850876460508, 'logit_5': -2.752043837310379, 'logit_6': -3.5353311026544043, 'logit_7': 2.9294091661994814, 'logit_8': 0.8089200939456704, 'logit_9': 1.664580622368364, 'logit_10': -3.8353240456335804}. Best is trial 0 with value: 0.3773051008529895.
[I 2026-07-30 09:27:03,256] Trial 1 finished with value: 0.3718194475540194 and parameters: {'logit_0': 3.7592788172959546, 'logit_1': 2.659541126403374, 'logit_2': -2.3012871145737908, 'logit_3': -2.545400262343195, 'logit_4': -2.5327639211725295, 'logit_5': -1.5660620563236982, 'logit_6': 0.19805145305790273, 'logit_7': -0.5444398508630739, 'logit_8': -1.6701668784156647, 'logit_9': 0.8948231577790358, 'logit_10': -2.8840491147836653}. Best is trial 0 with value: 0.3773051008529895.
[I 2026-07-30 09:27:04,600]

[I 2026-07-30 09:30:33,652] A new study created in memory with name: no-name-d5c12170-4bf2-4ef2-a6ed-c02a790d3d1b


...

[I 2026-07-30 09:40:21,544] A new study created in memory with name: no-name-5b4ce2bb-f96a-4b38-b68b-88e5962a3134


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-07-30 09:40:24,040] Trial 0 finished with value: 0.4012183766236799 and parameters: {'logit_0': -1.0036790492211, 'logit_1': 3.6057144512793293, 'logit_2': 1.8559515344912407, 'logit_3': 0.7892678735762928, 'logit_4': -2.751850876460508, 'logit_5': -2.752043837310379, 'logit_6': -3.5353311026544043, 'logit_7': 2.9294091661994814, 'logit_8': 0.8089200939456704, 'logit_9': 1.664580622368364, 'logit_10': -3.8353240456335804}. Best is trial 0 with value: 0.4012183766236799.
[I 2026-07-30 09:40:26,536] Trial 1 finished with value: 0.4019335095332853 and parameters: {'logit_0': 3.7592788172959546, 'logit_1': 2.659541126403374, 'logit_2': -2.3012871145737908, 'logit_3': -2.545400262343195, 'logit_4': -2.5327639211725295, 'logit_5': -1.5660620563236982, 'logit_6': 0.19805145305790273, 'logit_7': -0.5444398508630739, 'logit_8': -1.6701668784156647, 'logit_9': 0.8948231577790358, 'logit_10': -2.8840491147836653}. Best is trial 1 with value: 0.4019335095332853.
[I 2026-07-30 09:40:29,142]

,pr_auc,ndcg_at_50,precision_at_50,lift_at_50,fold,train_end,valid_start,valid_end,inner_best_objective
0,0.304624,0.769912,0.423704,2.512606,1,2023-05-19,2023-06-16,2023-10-16,0.442637
1,0.329374,0.789278,0.432533,2.483961,2,2024-02-14,2024-03-13,2024-07-15,0.431390
2,0.334475,0.756798,0.424267,2.266012,3,2024-11-07,2024-12-05,2025-04-08,0.467502
3,0.457166,0.818982,0.545476,1.761783,4,2025-08-04,2025-09-01,2025-12-30,0.451565


Outer PR-AUC instability: 0.05925730599937547


## 7. Kunci bobot, lalu buka final test 2026 sekali

Bobot final dioptimalkan hanya pada development set pra-2026. Setelah sel ini berjalan, jangan kembali mengubah target, fitur, split, atau hyperparameter berdasarkan hasil 2026.

In [8]:
final_inner = make_walk_forward_folds(development['date'], n_folds=5, min_train_months=18, valid_months=3)
final_study, final_weights = optimize_weights(development, final_inner, n_trials=N_TRIALS)
final_directions = learn_directions(development)

locked_model = pd.DataFrame({
    'feature': FEATURES,
    'rank_feature': RANK_FEATURES,
    'weight': final_weights,
    'direction': final_directions
}).sort_values('weight', ascending=False)
locked_model.to_csv(OUTPUT_DIR / 'locked_weights_pre2026.csv', index=False)
with open(OUTPUT_DIR / 'locked_model_metadata.json', 'w') as f:
    json.dump({'optimized_through': str(development.date.max().date()),
               'final_test_start': str(FINAL_TEST_START.date()),
               'best_inner_objective': final_study.best_value,
               'target_abnormal_return': TARGET_ABNORMAL_RETURN,
               'target_max_volume_ratio': TARGET_MAX_VOLUME_RATIO,
               'horizon': HORIZON, 'seed': SEED}, f, indent=2)
display(locked_model)

final_test = model_data[model_data['date'] >= FINAL_TEST_START].copy()
if final_test.empty:
    print('Belum ada baris 2026 dengan label horizon lengkap.')
else:
    final_scored = score_frame(final_test, final_weights, final_directions)
    final_metrics = pd.Series(evaluate(final_scored), name='2026 untouched test')
    display(final_metrics.to_frame())
    final_scored.to_parquet(OUTPUT_DIR / 'final_test_2026_scored.parquet', index=False)

[I 2026-07-30 09:47:10,409] A new study created in memory with name: no-name-c1cf2231-4fdb-4fb9-8f15-3bd75b1a0a6b


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-07-30 09:47:15,336] Trial 0 finished with value: 0.40613144744285684 and parameters: {'logit_0': -1.0036790492211, 'logit_1': 3.6057144512793293, 'logit_2': 1.8559515344912407, 'logit_3': 0.7892678735762928, 'logit_4': -2.751850876460508, 'logit_5': -2.752043837310379, 'logit_6': -3.5353311026544043, 'logit_7': 2.9294091661994814, 'logit_8': 0.8089200939456704, 'logit_9': 1.664580622368364, 'logit_10': -3.8353240456335804}. Best is trial 0 with value: 0.40613144744285684.
[I 2026-07-30 09:47:20,417] Trial 1 finished with value: 0.4137451477966839 and parameters: {'logit_0': 3.7592788172959546, 'logit_1': 2.659541126403374, 'logit_2': -2.3012871145737908, 'logit_3': -2.545400262343195, 'logit_4': -2.5327639211725295, 'logit_5': -1.5660620563236982, 'logit_6': 0.19805145305790273, 'logit_7': -0.5444398508630739, 'logit_8': -1.6701668784156647, 'logit_9': 0.8948231577790358, 'logit_10': -2.8840491147836653}. Best is trial 1 with value: 0.4137451477966839.
[I 2026-07-30 09:47:27,93

,feature,rank_feature,weight,direction
9,range_20,r_range_20,0.546172,1
8,turnover_accel,r_turnover_accel,0.246163,1
5,volatility_20,r_volatility_20,0.119976,1
10,drawdown_60,r_drawdown_60,0.050262,-1
7,volume_z_20,r_volume_z_20,0.021641,1
3,abret_5,r_abret_5,0.004253,1
0,mom_5,r_mom_5,0.003909,1
2,mom_60,r_mom_60,0.003396,1
6,volume_ratio_5_60,r_volume_ratio_5_60,0.003028,1
4,abret_20,r_abret_20,0.000893,1


,2026 untouched test
pr_auc,0.252129
ndcg_at_50,0.539188
precision_at_50,0.321488
lift_at_50,2.190012


## 8. Validation proxy: suspected suspension dalam 20 hari bursa

Karena riwayat UMA resmi belum tersedia, validasi tambahan memakai **proxy suspected suspension** dari sesi tidak aktif di Parquet. Event dimulai ketika ticker kehilangan bar atau mencatat volume nol minimal 3 sesi IHSG berturut-turut, dengan sesi aktif sebelum dan sesudah rangkaian tersebut. Ini bukan UMA resmi: pola serupa dapat berasal dari masalah Yahoo, saham sangat tidak likuid, atau perubahan ticker.

In [13]:
MIN_MISSING_SESSIONS = 3

def suspension_proxy_events(price_panel, benchmark_dates, min_missing=3):
    """Cari run bar hilang/volume nol yang diapit sesi bervolume positif."""
    calendar = pd.DatetimeIndex(sorted(pd.to_datetime(benchmark_dates).unique()))
    event_positions = {}
    event_rows = []

    for ticker, g in tqdm(price_panel.groupby('ticker'), desc='Detect suspension proxy', unit='ticker'):
        g = g[['date','volume']].drop_duplicates('date').set_index('date').sort_index()
        observed_dates = calendar.intersection(g.index)
        if len(observed_dates) < 2:
            event_positions[ticker] = np.array([], dtype=int)
            continue

        first = calendar.searchsorted(observed_dates.min())
        last = calendar.searchsorted(observed_dates.max())
        volume = g['volume'].reindex(calendar)
        inactive = (volume.isna() | volume.le(0)).to_numpy()
        inactive[:first + 1] = False
        inactive[last:] = False

        padded = np.r_[False, inactive, False]
        starts = np.flatnonzero(~padded[:-1] & padded[1:])
        ends = np.flatnonzero(padded[:-1] & ~padded[1:]) - 1
        confirmed = [(s, e) for s, e in zip(starts, ends)
                     if (e - s + 1) >= min_missing
                     and s > first and e < last
                     and pd.notna(volume.iloc[s - 1]) and volume.iloc[s - 1] > 0
                     and pd.notna(volume.iloc[e + 1]) and volume.iloc[e + 1] > 0]
        event_positions[ticker] = np.array([s for s, _ in confirmed], dtype=int)
        for start, end in confirmed:
            block = volume.iloc[start:end + 1]
            event_rows.append({
                'ticker': ticker,
                'proxy_start': calendar[start],
                'inactive_sessions': int(end - start + 1),
                'missing_bar_sessions': int(block.isna().sum()),
                'zero_volume_sessions': int(block.fillna(1).le(0).sum()),
                'last_active_before': calendar[start - 1],
                'first_active_after': calendar[end + 1]
            })
    event_columns = [
        'ticker', 'proxy_start', 'inactive_sessions', 'missing_bar_sessions',
        'zero_volume_sessions', 'last_active_before', 'first_active_after'
    ]
    return event_positions, pd.DataFrame(event_rows, columns=event_columns), calendar

def attach_future_suspension_proxy(scored, event_positions, calendar, horizon=20):
    out = scored.copy()
    labels = np.full(len(out), -1, dtype=np.int8)
    for i, (ticker, d) in enumerate(tqdm(zip(out.ticker, out.date), total=len(out), desc='Proxy labels')):
        pos = calendar.searchsorted(d)
        if pos >= len(calendar) or calendar[pos] != d or pos + horizon >= len(calendar):
            continue  # horizon belum observable
        events = event_positions.get(ticker, np.array([], dtype=int))
        left = np.searchsorted(events, pos + 1)
        labels[i] = int(left < len(events) and events[left] <= pos + horizon)
    out['suspected_suspension_next_20d'] = labels
    return out

if 'final_scored' not in globals():
    print('Final test belum tersedia.')
else:
    event_positions, proxy_events, trading_calendar = suspension_proxy_events(
        stocks[['ticker','date','volume']], bench['date'], MIN_MISSING_SESSIONS
    )
    proxy_events.to_csv(OUTPUT_DIR / 'suspected_suspension_events.csv', index=False)
    proxy_scored = attach_future_suspension_proxy(final_scored, event_positions, trading_calendar, HORIZON)
    observable = proxy_scored[proxy_scored.suspected_suspension_next_20d >= 0].copy()
    validation = observable.rename(columns={'target': 'price_volume_target'})
    validation['target'] = validation['suspected_suspension_next_20d']
    proxy_pr_auc = (average_precision_score(validation.target, validation.score)
                    if validation.target.nunique() > 1 else np.nan)
    proxy_topk = datewise_topk_metrics(validation).mean(numeric_only=True).to_dict()
    print({'proxy_suspension_pr_auc': proxy_pr_auc,
           'proxy_event_rate': validation.target.mean(),
           'detected_gap_events': len(proxy_events), **proxy_topk})
    if proxy_events.empty:
        print(f'Tidak ada confirmed gap minimal {MIN_MISSING_SESSIONS} sesi yang kemudian kembali diperdagangkan.')
    else:
        display(proxy_events.sort_values('proxy_start', ascending=False).head(20))

Detect suspension proxy:   0%|          | 0/958 [00:00<?, ?ticker/s]

Proxy labels:   0%|          | 0/108150 [00:00<?, ?it/s]

{'proxy_suspension_pr_auc': 0.0347809009880605, 'proxy_event_rate': 0.014450506774574013, 'detected_gap_events': 2851, 'precision_at_50': 0.05894736842105264, 'ndcg_at_50': 0.30687056571868876, 'lift_at_50': 3.668366892979939}


,ticker,proxy_start,inactive_sessions,missing_bar_sessions,zero_volume_sessions,last_active_before,first_active_after
470,BSWD,2026-07-20,6,0,6,2026-07-17,2026-07-28
953,FLMC,2026-07-03,10,0,10,2026-07-02,2026-07-17
2199,SINI,2026-07-01,4,0,4,2026-06-30,2026-07-07
2847,ZATA,2026-06-30,16,0,16,2026-06-29,2026-07-22
92,AIMS,2026-06-30,9,0,9,2026-06-29,2026-07-13
622,COAL,2026-06-30,12,0,12,2026-06-29,2026-07-16
313,BLTZ,2026-06-30,10,0,10,2026-06-29,2026-07-14
140,ASMI,2026-06-30,7,0,7,2026-06-29,2026-07-09
2611,TFCO,2026-06-29,3,0,3,2026-06-26,2026-07-02
1894,PDES,2026-06-26,4,0,4,2026-06-25,2026-07-02


## 9. Kandidat operasional terbaru

Semua ticker—termasuk IDX80/LQ45—mendapat skor. Filter kontrol baru diterapkan setelah scoring. Output final berisi maksimum 50 kandidat non-kontrol.

In [17]:
latest_date = panel.loc[panel[RANK_FEATURES].notna().any(axis=1), 'date'].max()
latest = panel[panel.date.eq(latest_date)].copy()
latest_scored = score_frame(latest, final_weights, final_directions)

def active_control_flags(rows, controls):
    if controls.empty:
        return pd.Series(False, index=rows.index)
    keys = controls[['ticker','effective_from','effective_to']].copy()
    merged = rows[['ticker','date']].reset_index().merge(keys, on='ticker', how='left')
    active = ((merged.effective_from.isna() | (merged.date >= merged.effective_from)) &
              (merged.effective_to.isna() | (merged.date <= merged.effective_to)) &
              merged.ticker.isin(set(keys.ticker)))
    return active.groupby(merged['index']).any().reindex(rows.index, fill_value=False)

latest_scored['is_idx80_or_lq45'] = active_control_flags(latest_scored, controls)
all_scores = latest_scored.sort_values('score', ascending=False)
candidates = all_scores[~all_scores.is_idx80_or_lq45].head(TOP_K).copy()
cols = ['date','ticker','sector','subsector','score','is_idx80_or_lq45'] + FEATURES
candidates[cols].to_csv(OUTPUT_DIR / f'candidates_{latest_date:%Y%m%d}.csv', index=False)
all_scores[['date','ticker','sector','subsector','score','is_idx80_or_lq45']].to_csv(
    OUTPUT_DIR / f'all_scores_{latest_date:%Y%m%d}.csv', index=False
)
print(f'Scoring date: {latest_date:%Y-%m-%d} | final candidates: {len(candidates)}')
display(candidates[cols[:6]].reset_index(drop=True))

Scoring date: 2026-07-30 | final candidates: 50


,date,ticker,sector,subsector,score,is_idx80_or_lq45
0,2026-07-30,MCAS,Teknologi,Perangkat Lunak & Jasa TI,0.979100,False
1,2026-07-30,MGNA,Barang Konsumen Non-Primer,Jasa Konsumen,0.960421,False
2,2026-07-30,MPRO,Properti & Real Estat,Properti & Real Estat,0.957295,False
3,2026-07-30,KDTN,Barang Konsumen Non-Primer,Jasa Konsumen,0.956690,False
4,2026-07-30,DMMX,Teknologi,Perangkat Lunak & Jasa TI,0.951431,False
5,2026-07-30,BAJA,Barang Baku,Barang Baku,0.946808,False
6,2026-07-30,MLPT,Teknologi,Perangkat Lunak & Jasa TI,0.943129,False
7,2026-07-30,COCO,Barang Konsumen Primer,Makanan & Minuman,0.940428,False
8,2026-07-30,ZONE,Barang Konsumen Non-Primer,Perdagangan Ritel,0.936922,False
9,2026-07-30,MDIA,Barang Konsumen Non-Primer,Media & Hiburan,0.929908,False


In [19]:
list(candidates['ticker'])

['MCAS',
 'MGNA',
 'MPRO',
 'KDTN',
 'DMMX',
 'BAJA',
 'MLPT',
 'COCO',
 'ZONE',
 'MDIA',
 'KOKA',
 'DWGL',
 'INAI',
 'ECII',
 'DOOH',
 'KOBX',
 'PSDN',
 'NTBK',
 'FUTR',
 'LUCY',
 'JGLE',
 'DEPO',
 'SQMI',
 'TAMA',
 'FLMC',
 'TNCA',
 'KBLV',
 'KLIN',
 'GDST',
 'KOTA',
 'PEGE',
 'SULI',
 'MMIX',
 'TFAS',
 'ZATA',
 'WOOD',
 'LAND',
 'KOPI',
 'HOPE',
 'FILM',
 'PTMP',
 'OILS',
 'TRUS',
 'ALDO',
 'DIVA',
 'LION',
 'RLCO',
 'NANO',
 'ELPI',
 'RONY']

## Checklist sebelum produksi

- Isi dan audit riwayat `control_universe.csv`; jangan hanya memakai snapshot jika backtest filter kontrol ikut dilaporkan.
- Audit `suspected_suspension_events.csv`; proxy gap Yahoo bukan bukti suspend atau UMA resmi.
- Periksa `outputs/missing_yfinance.csv`; delisting, IPO baru, ticker berubah, dan data volume nol perlu perlakuan eksplisit.
- Laporkan prevalence target, coverage universe, PR-AUC, Precision@50, Lift@50, serta interval kepercayaan bootstrap per tanggal.
- Jangan mengulang tuning setelah melihat final test 2026. Buat versi eksperimen baru bila definisi target atau fitur berubah.
- Skor adalah alat penyaringan riset, bukan rekomendasi investasi.